# 6. 평가
 모델 학습 단계에서 4-fold 교차검증의 F1로 선택한 설정과 임계값을 그대로 평가합니다. 평가자료를 보고 다시 선택하거나 재학습하지 않습니다.
 LR·OCSVM·RF의 혼동행렬, F1, 재현율, 정밀도, 오탐률, 점수 분포, 오류 입력을 저장하고 모두 정상/모두 위험 기준과 비교합니다. CN7의 EDA 규칙은 후향 참고 기준입니다. IF는 순위를 따로 기록하며 IF 이후 분류의 연결 효과로 해석하지 않습니다.
 실행 순서: 환경 → 변환 함수 → 모델 위치 → 평가 함수 → 실행 → 그림. CN7/RG3는 각각 별도 결과를 저장합니다. 기본 실행은 함수 정의와 경로 확인만 수행합니다.

## 6.1 환경 및 입력·지표 검증

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, uuid, os, copy, warnings, platform
import numpy as np
import pandas as pd
import joblib, sklearn
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path(r'C:\workspace\26KAMP')]
             if (p/'data/schema/input_features.json').is_file()), None)
if ROOT is None: raise FileNotFoundError('26KAMP 저장소 위치를 확인하세요.')
FEATURES = json.loads((ROOT/'data/schema/input_features.json').read_text(encoding='utf-8'))['feature_columns']
def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def digest(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def timestamp(): return datetime.now(timezone.utc).isoformat()
def run_id(prefix): return prefix+'-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'-'+uuid.uuid4().hex[:8]
def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    def convert(x):
        if isinstance(x,np.generic):return x.item()
        if isinstance(x,Path):return str(x)
        raise TypeError(type(x).__name__)
    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False,default=convert),encoding='utf-8')
    os.replace(tmp,path)
def validate_inputs(frame):
    if frame.empty or not set(FEATURES)<=set(frame): raise ValueError('입력 24개 변수 누락 또는 빈 자료')
    x=frame.loc[:,FEATURES].copy()
    if not all(pd.api.types.is_numeric_dtype(d) for d in x.dtypes):raise ValueError('입력은 숫자형이어야 합니다.')
    if not np.isfinite(x.to_numpy(dtype=float)).all():raise ValueError('결측 또는 무한값이 있습니다.')
    return x.astype(float)
def labels(values):
    y=pd.to_numeric(pd.Series(values),errors='raise').to_numpy()
    if not np.isfinite(y).all() or not set(y)<={0,1}:raise ValueError('라벨은 결측 없는 0/1이어야 합니다.')
    return y.astype(int)
def fingerprints(x):
    a=validate_inputs(x).to_numpy(dtype='<f8',copy=True);a[a==0]=0
    return [hashlib.sha256(row.tobytes()).hexdigest() for row in a]
def load_dataset(dataset):
    if dataset not in {'cn7','rg3'}:raise ValueError(dataset)
    folder=ROOT/f'data/processed/{dataset}/conservative'
    manifest=read_json(folder/'splits/split_manifest.json')
    for name,expected in manifest['source_sha256'].items():
        if digest(folder/name)!=expected:raise ValueError('분할 원본 해시 불일치: '+name)
    for name,expected in manifest['artifact_sha256'].items():
        if digest(folder/'splits'/name)!=expected:raise ValueError('분할 산출물 해시 불일치: '+name)
    x=pd.read_csv(folder/'X_labeled.csv',float_precision='round_trip')
    if list(x)!=FEATURES:raise ValueError('전처리 입력 순서가 스키마와 다릅니다.')
    x=validate_inputs(x);y=labels(pd.read_csv(folder/'y_labeled.csv').PassOrFail)
    split=pd.read_csv(folder/'splits/split_assignments.csv')
    if len(x)!=len(y) or not np.array_equal(split.pattern_row,np.arange(len(x))) or not np.array_equal(split.label,y):raise ValueError('분할/라벨 정렬 불일치')
    dev=split.loc[split.partition.eq('development'),'pattern_row'].to_numpy()
    test=split.loc[split.partition.eq('test'),'pattern_row'].to_numpy()
    if set(fingerprints(x.loc[dev]))&set(fingerprints(x.loc[test])):raise ValueError('개발/평가 입력 패턴 중복')
    return x,y,split,dev,test
def classification_metrics(y,p):
    y=labels(y);p=labels(p)
    if len(y)!=len(p):raise ValueError('정답과 예측 길이 불일치')
    tp=int(((y==1)&(p==1)).sum());fp=int(((y==0)&(p==1)).sum())
    fn=int(((y==1)&(p==0)).sum());tn=int(((y==0)&(p==0)).sum())
    return dict(TP=tp,FP=fp,FN=fn,TN=tn,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,
                recall=tp/(tp+fn) if tp+fn else None,precision=tp/(tp+fp) if tp+fp else None,
                FPR=fp/(fp+tn) if fp+tn else None)
def score_metrics(y,s):
    y=labels(y);s=np.asarray(s,dtype=float)
    return dict(AP=float(average_precision_score(y,s)) if y.sum() else None,
                ROC_AUC=float(roc_auc_score(y,s)) if len(set(y))==2 else None)

## 6.2 학습 모델의 변환 함수 보존
053 RF와 052 OCSVM 노트북의 함수 사본을 사용합니다. 이름이 바뀐 노트북을 가리키는 운영 경로에 의존하지 않습니다.

In [ ]:
# 학습/탐색 셀을 실행하지 않고 각 모델의 변환을 별도 공간에 보존합니다.
rf_library = dict(globals())
exec(compile(r'''
SEED = 42
SCENARIOS = ('raw_no_scaler', 'without_cycle', 'pressure', 'plasticizing', 'thermal', 'domain_all', 'group_pca')
GROUPS = {'충전과 전환': ['Injection_Time', 'Filling_Time', 'Cushion_Position', 'Max_Injection_Speed', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure'], '계량과 가소화': ['Plasticizing_Time', 'Plasticizing_Position', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Back_Pressure', 'Average_Back_Pressure'], '실린더와 호퍼 온도': [f'Barrel_Temperature_{i}' for i in range(1, 7)] + ['Hopper_Temperature'], '금형 온도': ['Mold_Temperature_3', 'Mold_Temperature_4'], '형체와 전체 주기': ['Clamp_Close_Time', 'Clamp_Open_Position', 'Cycle_Time']}
DOMAIN_SPEC = [dict(feature='사출_전환압력_제공값차', group='pressure', operation='difference', columns=['Max_Injection_Pressure', 'Max_Switch_Over_Pressure']), dict(feature='스크루RPM_제공값차', group='plasticizing', operation='difference', columns=['Max_Screw_RPM', 'Average_Screw_RPM']), dict(feature='배압_제공값차', group='plasticizing', operation='difference', columns=['Max_Back_Pressure', 'Average_Back_Pressure']), dict(feature='배럴_제공값평균', group='thermal', operation='mean', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)]), dict(feature='배럴_제공값센서간산포', group='thermal', operation='std', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)]), dict(feature='금형_제공값평균', group='thermal', operation='mean', columns=['Mold_Temperature_3', 'Mold_Temperature_4']), dict(feature='금형온도_제공값차', group='thermal', operation='difference', columns=['Mold_Temperature_3', 'Mold_Temperature_4'])]
SCENARIO_GROUPS = {'raw_no_scaler': (), 'without_cycle': (), 'pressure': ('pressure',), 'plasticizing': ('plasticizing',), 'thermal': ('thermal',), 'domain_all': ('pressure', 'plasticizing', 'thermal'), 'group_pca': ()}

def derived_frame(coordinates: pd.DataFrame, specs: list[dict]) -> pd.DataFrame:
    result = pd.DataFrame(index=coordinates.index)
    for spec in specs:
        values = coordinates[spec['active_columns']]
        if spec['operation'] == 'difference':
            result[spec['feature']] = values.iloc[:, 0] - values.iloc[:, 1]
        elif spec['operation'] == 'mean':
            result[spec['feature']] = values.mean(axis=1)
        elif spec['operation'] == 'std':
            result[spec['feature']] = values.std(axis=1, ddof=0)
        else:
            raise ValueError(spec['operation'])
    return result

def feature_frame(bundle: dict, inputs: pd.DataFrame) -> pd.DataFrame:
    inputs = inputs.loc[:, bundle['input_columns']]
    scenario = bundle['scenario']
    if scenario == 'raw_no_scaler' or scenario == 'without_cycle':
        return inputs.loc[:, bundle['base_columns']].copy()
    if bundle.get('feature_coordinates') == 'provided_v1':
        coordinates = inputs
    elif 'feature_coordinates' not in bundle and 'normal_scaler' in bundle:
        coordinates = pd.DataFrame(bundle['normal_scaler'].transform(inputs), index=inputs.index, columns=inputs.columns)
    else:
        raise ValueError('Unknown RF feature coordinates; retrain the model')
    if scenario == 'group_pca':
        result = pd.DataFrame(index=inputs.index)
        for group, item in bundle['pcas'].items():
            values = coordinates[item['columns']].to_numpy()
            scores = item['pca'].transform(values)
            for j in range(scores.shape[1]):
                result[f'{group}_PC{j + 1}'] = scores[:, j]
            if len(item['columns']) > scores.shape[1]:
                reconstruction = item['pca'].inverse_transform(scores)
                result[f'{group}_재구성RMSE'] = np.sqrt(np.mean((values - reconstruction) ** 2, axis=1))
        return result
    return pd.concat([inputs.loc[:, bundle['base_columns']], derived_frame(coordinates, bundle['specs'])], axis=1)

def fit_features(Xtrain: pd.DataFrame, ytrain: np.ndarray, scenario: str):
    assert scenario in SCENARIOS and set(ytrain) == {0, 1}
    normal = Xtrain.iloc[np.flatnonzero(ytrain == 0)]
    active = [column for column in Xtrain if normal[column].nunique() > 1]
    base = [column for column in Xtrain if scenario != 'without_cycle' or column not in GROUPS['형체와 전체 주기']]
    bundle = dict(scenario=scenario, input_columns=list(Xtrain.columns), base_columns=base, specs=[], skipped_specs=[], pcas={}, feature_coordinates='provided_v1')
    for spec in DOMAIN_SPEC:
        if spec['group'] not in SCENARIO_GROUPS[scenario]:
            continue
        cols = [column for column in spec['columns'] if column in active]
        if len(cols) < 2:
            bundle['skipped_specs'].append(dict(feature=spec['feature'], reason='fewer than 2 active sensors'))
        else:
            bundle['specs'].append({**spec, 'active_columns': cols})
    if scenario == 'group_pca':
        for group, columns in GROUPS.items():
            cols = [column for column in columns if column in active]
            if cols:
                bundle['pcas'][group] = dict(columns=cols, pca=PCA(n_components=min(2, len(cols)), svd_solver='full').fit(normal[cols].to_numpy()))
    frame = feature_frame(bundle, Xtrain)
    bundle['variance'] = VarianceThreshold(0.0).fit(frame)
    bundle['feature_names'] = list(frame.columns[bundle['variance'].get_support()])
    values = bundle['variance'].transform(frame)
    assert values.shape[1] > 0 and np.isfinite(values).all()
    return (bundle, values)

def transform_features(bundle: dict, inputs: pd.DataFrame) -> np.ndarray:
    values = bundle['variance'].transform(feature_frame(bundle, inputs))
    assert values.shape[1] == len(bundle['feature_names']) and np.isfinite(values).all()
    return values

def make_forest(params: dict) -> RandomForestClassifier:
    return RandomForestClassifier(criterion='gini', bootstrap=True, max_samples=None, min_samples_split=2, ccp_alpha=0.0, oob_score=False, warm_start=False, random_state=SEED, n_jobs=1, **params)

def risk_probability(forest: RandomForestClassifier, values: np.ndarray) -> np.ndarray:
    risk_col = np.flatnonzero(forest.classes_ == 1)
    assert len(risk_col) == 1
    p = forest.predict_proba(values)[:, risk_col[0]]
    assert p.shape == (len(values),) and np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()
    return p
''', '053.random_forest_feature_library', 'exec'), rf_library)

oc_library = dict(globals())
exec(compile(r'''
GROUPS = {'충전과 전환': ['Injection_Time', 'Filling_Time', 'Cushion_Position', 'Max_Injection_Speed', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure'], '계량과 가소화': ['Plasticizing_Time', 'Plasticizing_Position', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Back_Pressure', 'Average_Back_Pressure'], '실린더와 호퍼 온도': [f'Barrel_Temperature_{i}' for i in range(1, 7)] + ['Hopper_Temperature'], '금형 온도': ['Mold_Temperature_3', 'Mold_Temperature_4'], '형체와 전체 주기': ['Clamp_Close_Time', 'Clamp_Open_Position', 'Cycle_Time']}
SCENARIOS = {'raw': (), 'legacy_without_cycle': (), 'pressure': ('pressure',), 'plasticizing': ('plasticizing',), 'thermal': ('thermal',), 'domain_all': ('pressure', 'plasticizing', 'thermal'), 'group_pca': ()}
DOMAIN_SPEC = [dict(feature='사출_전환압력_상대편차', group='pressure', operation='difference', columns=['Max_Injection_Pressure', 'Max_Switch_Over_Pressure'], hypothesis='최대 사출압력과 전환압력이 정상 대비 함께 변하는지 확인', limitation='서로 다른 시점의 요약값이며 실제 압력 강하가 아님', source='ARBURG 64/2017 p27'), dict(feature='스크루RPM_상대편차', group='plasticizing', operation='difference', columns=['Max_Screw_RPM', 'Average_Screw_RPM'], hypothesis='가소화 회전수의 최대·평균 상대 변화 불일치', limitation='실제 RPM 폭이나 시간적 변동성으로 해석할 수 없음', source='Sensors 2022 22(13):4792; 제공 변수명'), dict(feature='배압_상대편차', group='plasticizing', operation='difference', columns=['Max_Back_Pressure', 'Average_Back_Pressure'], hypothesis='가소화 배압의 최대·평균 상대 변화 불일치', limitation='실제 배압 차이 또는 불량 인과 효과가 아님', source='Sensors 2022 22(13):4792; 제공 변수명'), dict(feature='배럴_상대수준평균', group='thermal', operation='mean', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)], hypothesis='활성 배럴 센서의 정상 대비 공통 온도 이동', limitation='평균 섭씨 온도 아님; 센서 위치·목표 온도는 미확인', source='제공 변수명; 공정 가설'), dict(feature='배럴_상대편차표준편차', group='thermal', operation='std', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)], hypothesis='활성 배럴 센서의 정상 대비 변화가 서로 다른 정도', limitation='물리적 온도 균일도·공간 구배 아님', source='제공 변수명; 공정 가설'), dict(feature='금형_상대수준평균', group='thermal', operation='mean', columns=['Mold_Temperature_3', 'Mold_Temperature_4'], hypothesis='금형 센서의 정상 대비 공통 온도 이동', limitation='센서가 상·하형인지 미확인; 평균 섭씨 온도 아님', source='제공 변수명; 공정 가설'), dict(feature='금형온도_상대편차', group='thermal', operation='difference', columns=['Mold_Temperature_3', 'Mold_Temperature_4'], hypothesis='금형 두 센서의 정상 대비 상대 변화 불일치', limitation='실제 온도차·냉각 효율 아님', source='제공 변수명; 공정 가설')]

def derived_frame(z, specs):
    result = pd.DataFrame(index=z.index)
    for spec in specs:
        values = z[spec['active_columns']]
        operation = spec['operation']
        if operation == 'difference':
            result[spec['feature']] = values.iloc[:, 0] - values.iloc[:, 1]
        elif operation == 'mean':
            result[spec['feature']] = values.mean(axis=1)
        elif operation == 'std':
            result[spec['feature']] = values.std(axis=1, ddof=0)
        else:
            raise ValueError(operation)
    return result

def unscaled_frame(bundle, inputs):
    inputs = inputs[bundle['input_columns']]
    z = pd.DataFrame(bundle['normal_scaler'].transform(inputs), index=inputs.index, columns=inputs.columns)
    if bundle['scenario'] == 'group_pca':
        frame = pd.DataFrame(index=inputs.index)
        for group, item in bundle['pcas'].items():
            values = z[item['columns']].to_numpy()
            scores = item['pca'].transform(values)
            for j in range(scores.shape[1]):
                frame[f'{group}_PC{j + 1}'] = scores[:, j]
            if len(item['columns']) > scores.shape[1]:
                frame[f'{group}_재구성RMSE'] = np.sqrt(np.mean((values - item['pca'].inverse_transform(scores)) ** 2, axis=1))
        return frame
    return pd.concat([z[bundle['base_columns']], derived_frame(z, bundle['specs'])], axis=1)

def fit_features(normal, scenario):
    if scenario not in SCENARIOS:
        raise ValueError(scenario)
    active = [c for c in normal if normal[c].nunique() > 1]
    scaler = StandardScaler().fit(normal)
    bundle = dict(scenario=scenario, input_columns=list(normal), normal_scaler=scaler, specs=[], skipped_specs=[], pcas={})
    for spec in DOMAIN_SPEC:
        if spec['group'] not in SCENARIOS[scenario]:
            continue
        cols = [c for c in spec['columns'] if c in active]
        if len(cols) < 2:
            bundle['skipped_specs'].append({'feature': spec['feature'], 'reason': '활성 센서 2개 미만'})
        else:
            bundle['specs'].append({**spec, 'active_columns': cols})
    bundle['base_columns'] = [c for c in active if scenario != 'legacy_without_cycle' or c not in GROUPS['형체와 전체 주기']]
    if scenario == 'group_pca':
        z = pd.DataFrame(scaler.transform(normal), columns=normal.columns, index=normal.index)
        for group, cols in GROUPS.items():
            cols = [c for c in cols if c in active]
            if cols:
                bundle['pcas'][group] = {'columns': cols, 'pca': PCA(n_components=min(2, len(cols)), svd_solver='full').fit(z[cols].to_numpy())}
    frame = unscaled_frame(bundle, normal)
    bundle['variance'] = VarianceThreshold(0).fit(frame)
    values = bundle['variance'].transform(frame)
    bundle['feature_scaler'] = StandardScaler().fit(values)
    bundle['feature_names'] = list(frame.columns[bundle['variance'].get_support()])
    return (bundle, transform_features(bundle, normal))

def transform_features(bundle, inputs):
    frame = unscaled_frame(bundle, inputs)
    values = bundle['feature_scaler'].transform(bundle['variance'].transform(frame))
    assert np.isfinite(values).all()
    return values

def fit_model(values, nu, multiplier):
    model = OneClassSVM(kernel='rbf', nu=float(nu), gamma=float(multiplier) / values.shape[1], tol=0.0001, max_iter=100000).fit(values)
    assert model.fit_status_ == 0, 'OCSVM 수렴 실패'
    return model
''', '052.ocsvm_feature_library', 'exec'), oc_library)

## 6.3 평가할 모델의 저장 위치
`RF_RUNS`는 053 노트북의 탐색 결과 경로입니다. 해당 탐색을 다른 폴더에 실행했다면 실제 산출물 위치로 바꾸세요. 모델이 없으면 중단하며 다른 RF로 대체하지 않습니다. 모델 생성 시 scikit-learn 버전과 일치하는 환경을 사용하세요.

In [ ]:
# RF는 053 노트북의 전체 탐색 산출물만 사용합니다.
# 다른 위치에 저장했다면 RF_RUNS만 실제 경로로 바꾸세요.
RF_RUNS = {ds: ROOT/f'output/random_forest_{ds}/rf_parameter_search_v1' for ds in ['cn7','rg3']}
# 현재 전체 탐색을 실행한 위치입니다. 저장소 경로에 결과가 있으면 그 경로를 우선 사용합니다.
RF_EXTERNAL_ROOT=Path(r'C:\Users\Playdata\Documents\Codex\2026-10-08\go\work\rf053_results')
for ds in RF_RUNS:
    if not (RF_RUNS[ds]/'model.joblib').exists() and (RF_EXTERNAL_ROOT/ds/'model.joblib').exists():
        RF_RUNS[ds]=RF_EXTERNAL_ROOT/ds
LR_RUNS = {
    'cn7': ROOT/'output/logistic/cn7/lr-20261003T040927Z-66e8d685',
    'rg3': ROOT/'output/logistic/rg3/lr-20261003T040950Z-8c8b16ec',
}
OCSVM_RUNS = {
    'cn7':ROOT/'runtime/cn7/models/ocsvm-6bb288e7f3ba464e',
    'rg3':ROOT/'runtime/rg3/models/ocsvm-657078ff64fd4278',
}
IF_RUNS = {
    'cn7':ROOT/'runtime/cn7/models/if-a7a2fcba54e54d58',
    'rg3':ROOT/'runtime/rg3/models/if-7e69cab8fdd04a17',
}
MODEL_FOLDERS = {'rf':RF_RUNS,'lr':LR_RUNS,'ocsvm':OCSVM_RUNS,'if':IF_RUNS}
SAVED_PREDICTIONS = {
    kind:{ds:Path(paths[ds])/'test_predictions.csv' for ds in ['cn7','rg3']}
    for kind,paths in {'rf':RF_RUNS,'lr':LR_RUNS}.items()
}
SAVED_PREDICTIONS['ocsvm']={ds:ROOT/f'output/report_visualizations/20261006_v4/{ds}_ocsvm_frozen_test.csv' for ds in ['cn7','rg3']}
def selected_info(dataset,kind):
    folder=Path(MODEL_FOLDERS[kind][dataset]);path=folder/'model.joblib'
    if not path.is_file():raise FileNotFoundError(f'{kind.upper()} 산출물 없음: {path}. MODEL_FOLDERS를 확인하세요.')
    sp=folder/'selection_manifest.json';metadata=read_json(sp if sp.exists() else folder/'manifest.json')
    if metadata.get('artifact_sha256') and digest(path)!=metadata['artifact_sha256']:raise ValueError('저장 모델 해시 불일치')
    if kind=='rf' and (metadata.get('setting_count')!=2268 or metadata.get('test_used_for_selection') is not False):raise ValueError('053 전체 탐색 선택 기록이 아닙니다.')
    return dict(dataset=dataset,kind=kind,folder=str(folder),model_sha256=digest(path),metadata=metadata)
def load_selected(dataset,kind):
    folder=Path(MODEL_FOLDERS[kind][dataset]);path=folder/'model.joblib'
    if not path.is_file():raise FileNotFoundError(f'{kind.upper()} 산출물 없음: {path}. 위 MODEL_FOLDERS의 경로를 확인하세요.')
    selection_path=folder/'selection_manifest.json';manifest_path=folder/'manifest.json'
    metadata=read_json(selection_path) if selection_path.exists() else read_json(manifest_path)
    expected=metadata.get('artifact_sha256')
    if expected and digest(path)!=expected:raise ValueError('저장 모델 해시 불일치')
    expected_version=metadata.get('sklearn')
    if expected_version and expected_version!=sklearn.__version__:raise RuntimeError(f'{kind}: 생성 환경 sklearn {expected_version}, 현재 {sklearn.__version__}')
    # 이 저장소에서 생성한 모델만 로드합니다.
    with warnings.catch_warnings():
        warnings.filterwarnings('error',category=sklearn.exceptions.InconsistentVersionWarning)
        artifact=joblib.load(path)
    if kind=='rf':
        if not {'bundle','forest','threshold'}<=set(artifact):raise ValueError('053 전체 탐색 RF의 저장 형식이 아닙니다.')
        if metadata.get('test_used_for_selection') is not False:raise ValueError('RF 설정 선택 기록을 확인하세요.')
        if artifact.get('selection_manifest_sha256')!=digest(selection_path):raise ValueError('RF 선택 기록 해시 불일치')
    elif kind=='lr' and artifact.get('selection_sha256')!=digest(selection_path):raise ValueError('LR 선택 기록 해시 불일치')
    card=dict(dataset=dataset,kind=kind,artifact=artifact,folder=str(folder),model_sha256=digest(path),metadata=metadata)
    return card
def predict_scores(card,x):
    x=validate_inputs(x);b=card['artifact'];kind=card['kind']
    if kind=='rf':values=rf_library['transform_features'](b['bundle'],x);model=b['forest'];s=rf_library['risk_probability'](model,values)
    elif kind=='lr':
        values=oc_library['transform_features'](b['domain'],x) if b.get('domain') else x
        model=b['model'];s=model.predict_proba(values)[:,int(np.flatnonzero(model.classes_==1)[0])]
    elif kind=='ocsvm':s=-b['model'].decision_function(oc_library['transform_features'](b['features'],x))
    elif kind=='if':s=-b['model'].score_samples(b['scaler'].transform(b['variance'].transform(x)))
    else:raise ValueError(kind)
    s=np.asarray(s,dtype=float)
    if len(s)!=len(x) or not np.isfinite(s).all():raise ValueError('모델 점수 길이/유한값 오류')
    return s
def threshold(card):
    if card['kind']=='if':return None
    return float(card['artifact']['threshold'])

In [ ]:
DATASETS=['cn7','rg3']
RUN_EVALUATION=False
EVALUATION_ROOT=ROOT/'output/evaluation'/run_id('evaluation')
display(pd.DataFrame([{'dataset':ds,'model':kind,'folder':str(paths[ds]),'available':(Path(paths[ds])/'model.joblib').exists()} for kind,paths in MODEL_FOLDERS.items() for ds in DATASETS]))

## 6.4 평가 및 오류 분석
기본은 저장 예측 검산입니다. 모델 생성 환경이 다른 LR·OCSVM과 RF를 무리하게 함께 재추론하지 않고 저장 점수에서 지표를 다시 계산합니다. `RECOMPUTE_SCORES=True`는 모델 생성 환경이 일치할 때 사용합니다. IF 순위는 저장 모델로 계산합니다. OOF는 설정 선택에 사용한 튜닝 결과이고 대회 평가자료는 이미 관찰한 후속 평가입니다. 위험 예측이 없으면 정밀도를 빈 값으로 표시합니다.

In [ ]:
RECOMPUTE_SCORES=False  # 기본: 저장 예측 검산 / True: 동일 모델 생성 환경에서 재추론
def evaluate_models(dataset,output_root):
    x,y,split,dev,test=load_dataset(dataset)
    out=Path(output_root)/dataset;out.mkdir(parents=True,exist_ok=True)
    xt=x.loc[test];yt=y[test];rows=[];sources={}
    for kind in ['lr','ocsvm','rf']:
        if RECOMPUTE_SCORES:
            card=load_selected(dataset,kind);s=predict_scores(card,xt);t=threshold(card);p=(s>t).astype(int)
        else:
            card=selected_info(dataset,kind);q=pd.read_csv(SAVED_PREDICTIONS[kind][dataset])
            if q.pattern_row.duplicated().any() or set(q.pattern_row)!=set(test):raise ValueError('저장 평가 예측의 입력 패턴 누락/중복')
            q=q.set_index('pattern_row').loc[test];np.testing.assert_array_equal(q.label,yt)
            s=q['probability' if 'probability' in q else 'score'].to_numpy();t=float(q.threshold.iloc[0])
            meta=card['metadata'];choice=meta.get('choice',meta)
            if 'threshold' not in choice or not np.allclose(q.threshold,float(choice['threshold'])):raise ValueError('선택 임계값과 저장 평가 예측 불일치')
            if not np.isfinite(s).all():raise ValueError('저장 점수가 유한값이 아닙니다.')
            p=(s>t).astype(int);np.testing.assert_array_equal(q.prediction,p)
        m=classification_metrics(yt,p);m.update(score_metrics(yt,s))
        rows.append(dict(dataset=dataset,model=kind,threshold=t,**m))
        pred=pd.DataFrame(dict(pattern_row=test,label=yt,score=s,threshold=t,prediction=p))
        pred.to_csv(out/f'{kind}_test_predictions.csv',index=False)
        errors=pred.loc[pred.label.ne(pred.prediction)].copy()
        errors['error_type']=np.where(errors.label.eq(1),'FN','FP')
        errors=errors.join(x,on='pattern_row');errors.to_csv(out/f'{kind}_error_cases.csv',index=False)
        oof_file=Path(card['folder'])/'selected_oof_predictions.csv'
        if oof_file.exists():
            oof=pd.read_csv(oof_file);target=dict(zip(dev,y[dev]))
            if oof.pattern_row.duplicated().any() or set(oof.pattern_row)!=set(dev):raise ValueError('OOF 패턴 누락/중복')
            if not np.array_equal(oof.label,oof.pattern_row.map(target)):raise ValueError('OOF 정답 불일치')
            if not np.allclose(oof.threshold if 'threshold' in oof else t,t):raise ValueError('OOF 임계값 불일치')
            oof['prediction']=(oof['probability' if 'probability' in oof else 'score']>t).astype(int)
            write_json(out/f'{kind}_oof_metrics.json',dict(status='튜닝에 사용한 교차검증 결과',**classification_metrics(oof.label,oof.prediction)))
        saved=SAVED_PREDICTIONS[kind][dataset]
        if saved.exists():
            q=pd.read_csv(saved).set_index('pattern_row').loc[test]
            np.testing.assert_array_equal(q.label,yt)
            np.testing.assert_array_equal(q.prediction,p)
            np.testing.assert_allclose(q['probability' if 'probability' in q else 'score'],s,rtol=1e-10,atol=1e-12)
        sources[kind]={k:card[k] for k in ['folder','model_sha256','metadata']}
    for name,p in [('all_normal',np.zeros(len(test),int)),('all_risk',np.ones(len(test),int))]:
        rows.append(dict(dataset=dataset,model=name,threshold=None,**classification_metrics(yt,p)))
    if dataset=='cn7':
        rule=((xt.Average_Back_Pressure<=.314806)&(xt.Mold_Temperature_3>.093154)).astype(int)
        rows.append(dict(dataset=dataset,model='EDA_rule_posthoc',threshold=None,**classification_metrics(yt,rule)))
    table=pd.DataFrame(rows);table.to_csv(out/'classification_comparison.csv',index=False)
    # IF는 분류 임계값 대신 순위 자체를 평가합니다. RF 상위 비율 평가와 혼합하지 않습니다.
    card=load_selected(dataset,'if');s=predict_scores(card,xt)
    ranked=pd.DataFrame(dict(pattern_row=test,label=yt,if_score=s)).sort_values(['if_score','pattern_row'],ascending=[False,True])
    ranked['if_rank']=np.arange(1,len(ranked)+1);ranked.to_csv(out/'if_test_ranking.csv',index=False)
    write_json(out/'if_ranking_metrics.json',dict(**score_metrics(yt,s),label='위험 이력 패턴',combined_pipeline_verified=False))
    sources['if']={k:card[k] for k in ['folder','model_sha256','metadata']}
    write_json(out/'evaluation_manifest.json',dict(dataset=dataset,created_at=timestamp(),python=platform.python_version(),sklearn=sklearn.__version__,mode='재추론' if RECOMPUTE_SCORES else '저장 예측 검산',
               split_sha256=digest(ROOT/f'data/processed/{dataset}/conservative/splits/split_assignments.csv'),
               status='이미 관찰한 대회 평가자료의 후속 평가',settings_chosen_by='4-fold pooled OOF F1',sources=sources))
    return table
def plot_evaluation(dataset,folder):
    os.environ.setdefault('MPLCONFIGDIR',str(Path(folder)/'matplotlib_cache'))
    import matplotlib.pyplot as plt
    plt.rcParams['font.family']='Malgun Gothic';plt.rcParams['axes.unicode_minus']=False
    folder=Path(folder)/dataset;fig,axes=plt.subplots(1,3,figsize=(14,4),layout='constrained')
    for ax,kind in zip(axes,['lr','ocsvm','rf']):
        q=pd.read_csv(folder/f'{kind}_test_predictions.csv')
        for label,color,mark in [(0,'steelblue','o'),(1,'firebrick','x')]:
            part=q[q.label.eq(label)];ax.scatter(label+np.linspace(-.12,.12,len(part)),part.score,color=color,marker=mark,s=22,alpha=.7)
        ax.axhline(q.threshold.iloc[0],color='gray',ls='--');ax.set_title(dataset.upper()+' '+kind.upper())
        ax.set_xticks([0,1],['정상','위험']);ax.set_ylabel('모델 점수')
    fig.savefig(folder/'label_scores.png',dpi=160);display(fig);plt.close(fig)
    table=pd.read_csv(folder/'classification_comparison.csv');fig,ax=plt.subplots(figsize=(9,4),layout='constrained')
    ax.bar(table.model,table.F1);ax.tick_params(axis='x',rotation=25);ax.set_ylabel('평가 F1');ax.set_ylim(0,1)
    fig.savefig(folder/'f1_comparison.png',dpi=160);display(fig);plt.close(fig)

## 6.5 평가 실행

In [ ]:
if RUN_EVALUATION:
    comparison=pd.concat([evaluate_models(ds,EVALUATION_ROOT) for ds in DATASETS],ignore_index=True)
    display(comparison)
    print('평가 결과:',EVALUATION_ROOT)
else:
    print('모델 위치를 확인한 뒤 RUN_EVALUATION=True로 실행하세요.')

## 6.6 점수 및 성능 그림

In [ ]:
if RUN_EVALUATION:
    for ds in DATASETS:plot_evaluation(ds,EVALUATION_ROOT)

## 저장 결과
제품별 하위 폴더에 모델별 평가 예측, 오류 사례, OOF 지표, 성능 비교표, IF 순위와 지표, 모델 출처·해시 및 그림을 저장합니다. 하이퍼파라미터 탐색은 5번 모델 학습 노트북에서 수행합니다.